# Social Data Analysis and Visualization (02806) - Assignment 1

## Group 70

| Parts                     | Contributor                 |
| ------------------------- | --------------------------- |
| Assignment 1.1, 1.2, 1.3: | Brynjar Bjarkason (s253549) |
| Assignment 1.4:           | Adam Ajane (s211048)        |
| Assignment 1.5:           | Sævar Reynisson (sxxxxxx)   |


## Assignment 1.1: Temporal Overview

## Assignment 1.2: Crime Profiles by Police District


## Assignment 1.3: Visualizing Distribtuons

## Assignment 1.4: Spatial Power Law

## Assignment 1.5: Regression and Correlation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from pathlib import Path

# Selected crimes for this part
selected_crimes = ['Assault', 'Robbery', 'Fraud', 'Prostitution']


def linear_fit_closed_form(x, y):
    """Fit y = a*x + b using the Week 4 closed-form equations."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)

    x_mean = x.mean()
    y_mean = y.mean()

    numerator = np.sum(x * y) - n * x_mean * y_mean
    denominator = np.sum(x * x) - n * x_mean * x_mean

    if denominator == 0:
        return np.nan, np.nan

    a = numerator / denominator
    b = y_mean - a * x_mean
    return a, b


def r2_from_fit(x, y, a, b):
    """R^2 = 1 - SSE/SST for y against y_hat."""
    if not np.isfinite(a) or not np.isfinite(b):
        return np.nan

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    y_hat = a * x + b

    sse = np.sum((y - y_hat) ** 2)
    sst = np.sum((y - y.mean()) ** 2)

    if sst == 0:
        return np.nan

    return 1 - sse / sst


# Resolve data path
data_candidates = [
    Path('../data/sf_crime_merged_focus_2003_2025.csv'),
    Path('data/sf_crime_merged_focus_2003_2025.csv'),
]

for candidate in data_candidates:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError('Could not find sf_crime_merged_focus_2003_2025.csv in expected locations.')


# Build 168-hour weekly vectors per crime
df = pd.read_csv(data_path, usecols=['Focus Crime', 'Incident Date', 'Incident Time'])
df = df[df['Focus Crime'].isin(selected_crimes)].copy()

df['Incident Date'] = pd.to_datetime(df['Incident Date'], errors='coerce')
hour_str = df['Incident Time'].astype(str).str.extract(r'^(\d{1,2})')[0]
df['Hour'] = pd.to_numeric(hour_str, errors='coerce')
df = df.dropna(subset=['Incident Date', 'Hour'])
df['Hour'] = df['Hour'].astype(int).clip(0, 23)

df['HourOfWeek'] = df['Incident Date'].dt.dayofweek * 24 + df['Hour']

hourly_168 = (
    df.groupby(['HourOfWeek', 'Focus Crime'])
    .size()
    .rename('Count')
    .reset_index()
)

pivot = (
    hourly_168.pivot(index='HourOfWeek', columns='Focus Crime', values='Count')
    .reindex(range(168))
    .fillna(0)
    .reindex(columns=selected_crimes, fill_value=0)
)


# Pairwise scatterplots + regression + R^2
pairs = list(combinations(selected_crimes, 2))
n_pairs = len(pairs)
n_cols = 3
n_rows = int(np.ceil(n_pairs / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.atleast_1d(axes).ravel()
fig.suptitle('Assignment 1.5: Pairwise Weekly Rhythm Fits for Selected Crimes', fontsize=14, y=1.02)

rows = []

for ax, (crime_x, crime_y) in zip(axes, pairs):
    x = pivot[crime_x].to_numpy()
    y = pivot[crime_y].to_numpy()

    a, b = linear_fit_closed_form(x, y)
    r2 = r2_from_fit(x, y, a, b)

    if np.std(x) == 0 or np.std(y) == 0:
        r = np.nan
    else:
        r = np.corrcoef(x, y)[0, 1]

    rows.append({
        'Crime A': crime_x,
        'Crime B': crime_y,
        'a_slope': a,
        'b_intercept': b,
        'r': r,
        'R2': r2,
    })

    ax.scatter(x, y, s=18, alpha=0.6, color='tab:blue')

    if np.isfinite(a) and np.isfinite(b):
        x_line = np.linspace(x.min(), x.max(), 100)
        y_line = a * x_line + b
        ax.plot(x_line, y_line, color='black', linewidth=1.5)

    ax.text(
        0.03,
        0.97,
        f'r={r:.3f}\n$R^2$={r2:.3f}',
        transform=ax.transAxes,
        va='top',
        fontsize=9,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.85),
    )

    ax.set_title(f'{crime_x} vs {crime_y}')
    ax.set_xlabel(crime_x)
    ax.set_ylabel(crime_y)
    ax.grid(alpha=0.2)

for ax in axes[n_pairs:]:
    ax.axis('off')

plt.tight_layout()
plt.show()


# Ranking and short written answer
results = pd.DataFrame(rows).sort_values('R2', ascending=False).reset_index(drop=True)
most_correlated = results.iloc[0]
least_correlated = results.iloc[-1]

print('Pairwise results (sorted by R^2):')
print(
    results[['Crime A', 'Crime B', 'r', 'R2', 'a_slope', 'b_intercept']]
    .to_string(index=False, float_format=lambda v: f'{v:.3f}')
)

print('\nMost correlated pair:')
print(
    f"- {most_correlated['Crime A']} vs {most_correlated['Crime B']} "
    f"| r={most_correlated['r']:.3f}, R^2={most_correlated['R2']:.3f}"
)

print('\nLeast correlated pair:')
print(
    f"- {least_correlated['Crime A']} vs {least_correlated['Crime B']} "
    f"| r={least_correlated['r']:.3f}, R^2={least_correlated['R2']:.3f}"
)

print('\nInterpretation:')
print('- A higher R^2 means the two crime types share a similar weekly rhythm across the 168 hours.')
print('- A lower R^2 (close to 0) means their weekly timing is largely out of sync.')
print('- If crime concentrates in specific places/times, neighborhood averages can hide hotspots and risk windows.')
